# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper labels pages as growing or declining using trend direction, which is based on the change in traffic over the last 30 days compared with the previous 30 days. “Up” means more than 10% growth, while “Down” means more than 10% decline. The paper finds that growing pages were younger on average than declining pages (185 vs. 228 days). This validation supports an observed association between content age and growth, but it does not establish causality. A useful methodology question is whether other factors related to page age could also explain part of the difference.

### Finding 4 — The Freshness Multiplier

The outcome label again comes from whether pages are growing or declining, while freshness is based on how recently a page was updated. The paper reports a 5.43:1 growth-to-decline ratio for pages updated 31–90 days ago and also reports higher performance for refreshed older pages. These comparisons support an observed relationship between freshness and performance, but they do not by themselves prove that refreshing a page caused the improvement. A useful methodology question is whether refreshed and unrefreshed pages were comparable before the refresh.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [11]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [12]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

months = sorted({
    f.split("month=")[1].split("/")[0]
    for f in files
    if "fact_content_daily_performance/month=" in f
})

print("Available months:")
print(months)


Available months:
['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


In [13]:
from huggingface_hub import hf_hub_download

months_to_load = ["2026-01", "2026-02", "2026-03"]

monthly_paths = {}

for month in months_to_load:
    path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={month}/data_0.parquet",
        token=HF_TOKEN
    )
    monthly_paths[month] = path

print("Downloaded months:")
print(list(monthly_paths.keys()))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded months:
['2026-01', '2026-02', '2026-03']


In [15]:
import pandas as pd

required_columns = [
    "content_hash_id",
    "client_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec"
]

dfs = []

for month, path in monthly_paths.items():
    month_df = pd.read_parquet(
        path,
        columns=required_columns
    )

    print(month, month_df.shape)
    dfs.append(month_df)

df_time = pd.concat(dfs, ignore_index=True)

print("Combined shape:", df_time.shape)
print("Date range:", df_time["report_date"].min(), "to", df_time["report_date"].max())

2026-01 (7890817, 9)
2026-02 (7355108, 9)
2026-03 (9841378, 9)
Combined shape: (25087303, 9)
Date range: 2026-01-01 to 2026-03-31


In [17]:
# Recreate the same features used in Week 5
import numpy as np


df_time["ctr"] = np.where(
    df_time["gsc_impressions"] > 0,
    df_time["gsc_clicks"] / df_time["gsc_impressions"],
    0
)

df_time["engagement_rate"] = np.where(
    df_time["ga4_sessions"] > 0,
    df_time["ga4_engaged_sessions"] / df_time["ga4_sessions"],
    0
)

df_time["avg_engagement_sec"] = np.where(
    df_time["ga4_sessions"] > 0,
    df_time["ga4_total_engagement_sec"] / df_time["ga4_sessions"],
    0
)

df_time["position_bucket"] = pd.cut(
    df_time["gsc_avg_position"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=False
)

df_time["log_impressions"] = np.log1p(
    df_time["gsc_impressions"]
)

# Same Week-5 filtering rule
df_time = df_time[
    df_time["gsc_impressions"] >= 10
].reset_index(drop=True)

print("Shape after filtering:", df_time.shape)
print("Date range:", df_time["report_date"].min(), "to", df_time["report_date"].max())

print("\nMissing values in selected features:")
print(
    df_time[
        [
            "gsc_impressions",
            "log_impressions",
            "gsc_avg_position",
            "engagement_rate",
            "avg_engagement_sec",
            "position_bucket"
        ]
    ].isna().sum()
)

Shape after filtering: (5017480, 14)
Date range: 2026-01-01 to 2026-03-31

Missing values in selected features:
gsc_impressions           0
log_impressions           0
gsc_avg_position          1
engagement_rate           0
avg_engagement_sec        0
position_bucket       34523
dtype: int64


In [18]:
selected_features = [
    "gsc_impressions",
    "log_impressions",
    "gsc_avg_position",
    "engagement_rate",
    "avg_engagement_sec",
    "position_bucket"
]

before_rows = len(df_time)

df_time = df_time.dropna(
    subset=selected_features + ["ctr"]
).reset_index(drop=True)

print("Rows before removing missing values:", before_rows)
print("Rows after removing missing values:", len(df_time))
print("Rows removed:", before_rows - len(df_time))

print("\nRemaining missing values:")
print(df_time[selected_features + ["ctr"]].isna().sum())

Rows before removing missing values: 5017480
Rows after removing missing values: 4982957
Rows removed: 34523

Remaining missing values:
gsc_impressions       0
log_impressions       0
gsc_avg_position      0
engagement_rate       0
avg_engagement_sec    0
position_bucket       0
ctr                   0
dtype: int64


In [20]:
from datetime import date

train_mask = df_time["report_date"] < date(2026, 3, 1)
test_mask = df_time["report_date"] >= date(2026, 3, 1)

X_train_time = df_time.loc[
    train_mask, selected_features
].copy()

X_test_time = df_time.loc[
    test_mask, selected_features
].copy()

y_train_time = df_time.loc[
    train_mask, "ctr"
].copy()

y_test_time = df_time.loc[
    test_mask, "ctr"
].copy()

groups_train_time = df_time.loc[
    train_mask, "client_hash_id"
]

groups_test_time = df_time.loc[
    test_mask, "client_hash_id"
]

print("Train rows:", len(X_train_time))
print("Test rows:", len(X_test_time))

print("\nTrain date range:")
print(
    df_time.loc[train_mask, "report_date"].min(),
    "to",
    df_time.loc[train_mask, "report_date"].max()
)

print("\nTest date range:")
print(
    df_time.loc[test_mask, "report_date"].min(),
    "to",
    df_time.loc[test_mask, "report_date"].max()
)

print("\nTrain clients:", groups_train_time.nunique())
print("Test clients:", groups_test_time.nunique())

overlap = set(groups_train_time) & set(groups_test_time)

print("Clients appearing in both:", len(overlap))

Train rows: 2842376
Test rows: 2140581

Train date range:
2026-01-01 to 2026-02-28

Test date range:
2026-03-01 to 2026-03-31

Train clients: 41
Test clients: 44
Clients appearing in both: 37


In [21]:
from xgboost import XGBRegressor

time_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

time_model.fit(
    X_train_time,
    y_train_time
)

print("Time-aware model training completed successfully.")

Time-aware model training completed successfully.


In [22]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    ndcg_score
)
import numpy as np

time_predictions = time_model.predict(X_test_time)

time_mae = mean_absolute_error(
    y_test_time,
    time_predictions
)

time_rmse = np.sqrt(
    mean_squared_error(
        y_test_time,
        time_predictions
    )
)

time_r2 = r2_score(
    y_test_time,
    time_predictions
)

time_ndcg = ndcg_score(
    [y_test_time.values],
    [time_predictions]
)

print("Time-aware results:")
print("MAE :", time_mae)
print("RMSE:", time_rmse)
print("R²  :", time_r2)
print("NDCG:", time_ndcg)

Time-aware results:
MAE : 0.004400748527707436
RMSE: 0.009578834750672536
R²  : 0.021247495600057897
NDCG: 0.8064575984088161


### Week-5 model: before vs. time-aware evaluation

I re-evaluated the same Week-5 XGBoost model using a time-aware split. The original Week-5 evaluation used a grouped client split, while the new evaluation trained on January–February 2026 and tested on March 2026. A time-aware evaluation is useful here because the model is evaluated on a future period rather than randomly mixing observations across time. :contentReference[oaicite:0]{index=0}

| Metric | Week-5 grouped split | Time-aware split |
|---|---:|---:|
| MAE | 0.001552 | 0.004401 |
| RMSE | 0.013821 | 0.009579 |
| R² | -0.1237 | 0.02125 |
| NDCG | 0.6947 | 0.80646 |

The time-aware evaluation produced a higher NDCG (0.8065 vs. 0.6947), lower RMSE (0.00958 vs. 0.01382), and a slightly positive R² compared with the original grouped evaluation. MAE was higher (0.00440 vs. 0.00155).

These results show that the measured performance changes when the evaluation design changes. The time-aware result is more aligned with a future-period deployment scenario, but the two evaluations also use different training windows, so the difference should not be interpreted as a pure effect of the split strategy alone.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [23]:
## Section 3 — Leakage audit

target = "ctr"

audit_features = [
    "gsc_impressions",
    "log_impressions",
    "gsc_avg_position",
    "engagement_rate",
    "avg_engagement_sec",
    "position_bucket"
]

print("Target:", target)
print("\nFinal model features:")
for feature in audit_features:
    print("-", feature)

print("\nDirect target leakage:")
print("Target included in features:", target in audit_features)

print("\nFeature construction:")
print("ctr = clicks / impressions")
print("engagement_rate = engaged_sessions / sessions")
print("avg_engagement_sec = total_engagement_sec / sessions")
print("log_impressions = log1p(impressions)")
print("position_bucket = bucketized avg_position")


Target: ctr

Final model features:
- gsc_impressions
- log_impressions
- gsc_avg_position
- engagement_rate
- avg_engagement_sec
- position_bucket

Direct target leakage:
Target included in features: False

Feature construction:
ctr = clicks / impressions
engagement_rate = engaged_sessions / sessions
avg_engagement_sec = total_engagement_sec / sessions
log_impressions = log1p(impressions)
position_bucket = bucketized avg_position


In [24]:
# Check for temporal leakage

print("Temporal leakage audit:")
print()

print("1. gsc_impressions:")
print("   Current-day search impressions — no future data used.")

print("2. log_impressions:")
print("   Derived only from gsc_impressions — no additional information used.")

print("3. gsc_avg_position:")
print("   Current-day average search position — no future data used.")

print("4. engagement_rate:")
print("   Derived from current-day engaged sessions / sessions — no future data used.")

print("5. avg_engagement_sec:")
print("   Derived from current-day engagement time / sessions — no future data used.")

print("6. position_bucket:")
print("   Derived from current-day gsc_avg_position — no future data used.")

print()
print("Temporal leakage found: No, based on feature construction.")

Temporal leakage audit:

1. gsc_impressions:
   Current-day search impressions — no future data used.
2. log_impressions:
   Derived only from gsc_impressions — no additional information used.
3. gsc_avg_position:
   Current-day average search position — no future data used.
4. engagement_rate:
   Derived from current-day engaged sessions / sessions — no future data used.
5. avg_engagement_sec:
   Derived from current-day engagement time / sessions — no future data used.
6. position_bucket:
   Derived from current-day gsc_avg_position — no future data used.

Temporal leakage found: No, based on feature construction.


In [25]:
# Check content overlap between train and test

train_content_ids = set(
    df_time.loc[train_mask, "content_hash_id"]
)

test_content_ids = set(
    df_time.loc[test_mask, "content_hash_id"]
)

content_overlap = train_content_ids.intersection(
    test_content_ids
)

print("Train unique content:", len(train_content_ids))
print("Test unique content:", len(test_content_ids))
print("Content appearing in both:", len(content_overlap))
print(
    "Content overlap rate:",
    len(content_overlap) / len(test_content_ids)
)

Train unique content: 94910
Test unique content: 109593
Content appearing in both: 81262
Content overlap rate: 0.7414889637111859


In [26]:
# Check ID usage and client overlap

id_features = [
    "content_hash_id",
    "client_hash_id"
]

model_features = selected_features

print("ID leakage audit:")
print()

for feature in id_features:
    print(
        f"{feature} used as model feature:",
        feature in model_features
    )

train_clients = set(
    df_time.loc[train_mask, "client_hash_id"]
)

test_clients = set(
    df_time.loc[test_mask, "client_hash_id"]
)

client_overlap = train_clients.intersection(test_clients)

print()
print("Train unique clients:", len(train_clients))
print("Test unique clients:", len(test_clients))
print("Clients appearing in both:", len(client_overlap))
print(
    "Client overlap rate:",
    len(client_overlap) / len(test_clients)
)

ID leakage audit:

content_hash_id used as model feature: False
client_hash_id used as model feature: False

Train unique clients: 41
Test unique clients: 44
Clients appearing in both: 37
Client overlap rate: 0.8409090909090909


### Leakage audit

I audited the final feature set for direct target leakage, temporal leakage, identifier leakage, and train/test contamination.

- **Direct target leakage:** No. `ctr` is not included in the model features.
- **Temporal leakage:** No identified. The engineered features use only the corresponding day's GSC/GA4 measurements. No future-derived variables such as `trend_direction` or `trend_pct` were used.
- **Identifier leakage:** No. Neither `content_hash_id` nor `client_hash_id` is used as a model feature; they are used only for grouping and split analysis.
- **Content overlap:** 81,262 of 109,593 test content IDs (74.1%) also appear in the training period. This is expected in the time-aware setup because the model is evaluated on later observations of content that may have existed previously. Since the content ID itself is not a feature, this does not provide direct identifier leakage.
- **Client overlap:** 37 of 44 test clients (84.1%) also appear in the training period. This is expected for a time-aware split, where the goal is to evaluate performance on future observations from the same clients.

Overall, I found no direct feature leakage in the final model. The main limitation is that the time-aware evaluation allows the same content and clients to appear across training and test periods, so the result represents **future-period prediction for existing content and clients**, rather than generalization to completely unseen content or clients.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim:**

> The model can predict which content will perform better based on its search and engagement signals.

**Safer version:**

> The model measured the ability of current search and engagement signals to support directional ranking of content performance. Under the time-aware evaluation, the model achieved an NDCG of 0.8065 on the March 2026 test period. This result provides decision-support evidence for ranking content by expected CTR, but it does not establish that these signals cause future performance or guarantee individual content outcomes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.